# Human Pose / Body Posture Module — Social Cue Analysis for HRI

**Group:** The Unfiltered — Project 6 (Social cue analysis for Human-Robot Interaction)
**This notebook covers:** Ashkan Bazrgary's part — body posture / confidence recognition from body landmarks.

Training data: [Confidence Detection Dataset](https://www.kaggle.com/datasets/muhammadkhubaibahmad/confidence-detection-dataset)
on Kaggle (5,950 rows, 19 landmark-derived features + `confidence_label` target) — self-recording
video clips wasn't feasible (broken webcam + team scheduling), so training uses this public dataset
instead, with a separate MediaPipe extractor built for the eventual live-webcam demo.


## 1. Setup

In [ ]:
!pip install -q kaggle mediapipe opencv-python-headless scikit-learn pandas matplotlib joblib

import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay


## 2. Download the Kaggle dataset

You need a Kaggle API token: Kaggle -> profile -> Account -> "Create New Token" downloads `kaggle.json`.
Upload that file when prompted below (one-time per Colab session).


In [ ]:
from google.colab import files

print('Upload your kaggle.json (Kaggle -> Account -> Create New Token):')
uploaded = files.upload()

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json


In [ ]:
!kaggle datasets download -d muhammadkhubaibahmad/confidence-detection-dataset -p /content/confidence_data --unzip

DATA_DIR = '/content/confidence_data'
for root, dirs, fnames in os.walk(DATA_DIR):
    for fn in fnames:
        print(os.path.join(root, fn))


## 3. Inspect the data

Confirmed from the dataset page: `confidence_dataset.csv`, 5,950 rows, 19 features + target
`confidence_label`. Features are ratios/angles/distances computed from pose landmarks (the same
kind of engineering as our own `extract_features()` in section 6). Three columns are categorical:
`head_direction`, `arm_position`, `posture` — encoded in section 4, not dropped.


In [ ]:
csv_paths = glob.glob(os.path.join(DATA_DIR, '**', '*.csv'), recursive=True)
print('CSV files found:', csv_paths)

dfs = {p: pd.read_csv(p) for p in csv_paths}
for p, d in dfs.items():
    print('\n===', p, '===')
    print('shape:', d.shape)
    print('columns:', list(d.columns))
    print(d.head(3))


In [ ]:
# confirmed from the dataset page; double check against the printed columns above
MAIN_CSV = csv_paths[0]
LABEL_COL = 'confidence_label'
CATEGORICAL_COLS = ['head_direction', 'arm_position', 'posture']

df = dfs[MAIN_CSV]
print(df[LABEL_COL].value_counts())
print('\nnull counts:\n', df.isnull().sum())


## 4. Feature preparation

Numeric columns are used directly; the three categorical columns are one-hot encoded rather than
dropped, since they're part of the dataset's actual signal (not just numeric landmark ratios).


In [ ]:
numeric_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c != LABEL_COL]
present_cat_cols = [c for c in CATEGORICAL_COLS if c in df.columns]

X_numeric = df[numeric_cols].fillna(0)
X_categorical = pd.get_dummies(df[present_cat_cols].astype(str)) if present_cat_cols else pd.DataFrame(index=df.index)
X = pd.concat([X_numeric, X_categorical], axis=1)
feature_cols = list(X.columns)
print('Using', len(feature_cols), 'feature columns (', len(numeric_cols), 'numeric +',
      X_categorical.shape[1], 'one-hot from', present_cat_cols, ')')

le = LabelEncoder()
y = le.fit_transform(df[LABEL_COL])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
